# HengSeng Sentiment Price Prediction Agent — Research Notebook

**CrowdWisdomTrading internship assessment · Hong Kong equities · research only**

This notebook **does not claim a trained model or accurate forecast** until you run it with genuine licensed/approved social history and price data. The required last 10 days of sentiment are not enough to prove reliable daily out-of-sample performance.

Main path: **Apify LIHKG → (authorized Futu CLI, if permitted) → Exa News → daily HK prices → multilingual sentiment → UTC close alignment → ML walk-forward validation → visual report**. The second source is not yet Apify-based; see `docs/SOURCES.md` before submission.

## 0 — Setup

Install dependencies at repository root:
```bash
python -m pip install -e '.[test,sentiment]'
```

Fill `.env` locally with your **own** Apify and Exa tokens. **Never** paste credentials into notebook cells or push them to GitHub. With the booleans below left `False`, this notebook stays offline and uses previously collected CSVs only.

In [ ]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from news_predictor.ingest import (fetch_daily_prices, fetch_exa_news, run_lihkg_actor, run_futu_community, write_frame)
from news_predictor.sentiment import label_sentiment
from news_predictor.features import build_features, eligible_training_rows
from news_predictor.model import train_evaluate, save_artifacts, predict_next
from news_predictor.report import render_html, render_leaderboard, summarize_ticker

# If opened from the notebooks/ directory, locate repo root.
ROOT = Path.cwd()
if not (ROOT / "config/assets.json").exists(): ROOT = ROOT.parent
assert (ROOT / "config/assets.json").exists(), "Open notebook from project folder."
load_dotenv(ROOT / ".env")
ASSETS = {a["ticker"]: a for a in json.loads((ROOT / "config/assets.json").read_text(encoding="utf-8"))}
print('Supported tickers:', list(ASSETS))

## 1 — Choose your stock and source permissions

Ticker/stock links are **lookup inputs**, not ML features. Set one identifier for reproducible collection. Live collection can cost money, invokes public/third-party services, and requires permission to reuse the source data.

- `ENABLE_LIVE_COLLECTION = True` downloads market bars, invokes the LIHKG third-party Apify Actor, and searches Exa.
- `ENABLE_AUTHORIZED_FUTU = True` additionally invokes the official authenticated Futu CLI. **Its outputs need first-run schema validation and it is not an Apify Actor.**
- Keep both disabled for offline evaluation of *your own previously collected* CSVs.

In [ ]:
SELECTED_TICKER = "0700.HK"  # Change to any listed HKEX ticker
ENABLE_LIVE_COLLECTION = False
ENABLE_AUTHORIZED_FUTU = False    # Requires Futu CLI, OAuth login and explicit rights review
LOOKBACK_DAYS = 10
PRICE_START = "2025-01-01"

asset = ASSETS[SELECTED_TICKER]
raw_dir = ROOT / "data" / "raw" / SELECTED_TICKER
out_dir = ROOT / "outputs" / SELECTED_TICKER
print('Selected:',asset['name'], SELECTED_TICKER)

## 2 — Collect genuine source data (never fabricate missing posts)

The LIHKG Apify Actor documents `keywords`, `dateFrom`, `created_at`, `own_text`, and `like_count`. Futu publishes an official community-search CLI, but its actual JSON shape and historical access have **not been demonstrated** here. Exa provides published-date-filtered news results. `yfinance` is a research-only price data source, not an exchange-licensed production feed.

In [ ]:
if ENABLE_LIVE_COLLECTION:
    prices = fetch_daily_prices(asset, start=PRICE_START)
    lihkg, lihkg_meta = run_lihkg_actor(asset, lookback_days=LOOKBACK_DAYS)
    if ENABLE_AUTHORIZED_FUTU:
        futu, futu_meta = run_futu_community(asset, enabled=True)
    else:
        futu = pd.DataFrame(columns=lihkg.columns)
        futu_meta = {'status':'disabled_pending_permission_and_schema_review'}
    news = fetch_exa_news(asset, lookback_days=LOOKBACK_DAYS)
    social = pd.concat([lihkg, futu], ignore_index=True)
    if social.empty:
        raise RuntimeError('No usable timestamped social posts; inspect your Actor run')
    for filename, frame in [('prices.csv',prices),('social.csv',social),('news.csv',news)]:
        write_frame(frame, raw_dir / filename)
    raw_dir.mkdir(parents=True,exist_ok=True)
    (raw_dir/'provenance.json').write_text(json.dumps({'apify_lihkg':lihkg_meta,'futu':futu_meta,
        'exa_count':len(news),'market_provider':'yfinance_research'},indent=2))
    print('New data collected:', len(prices),'price bars;',len(social),'social posts;',len(news),'news items')
else:
    print('Offline mode: no network services called; existing CSVs required for next step.')

## 3 — Load, inspect and label multilingual text

Text sentiment is **not the trained stock price regressor**. A documented pretrained multilingual classifier assigns `P(positive)-P(negative)` to each social/news item (range [-1,1]). It is *not calibrated to Hong Kong finance*: annotate a sample manually before treating sentiment scores as accurate. Publication timestamps must be present; retrieval timestamps must never masquerade as publish times.

In [ ]:
required = ['prices.csv','social.csv','news.csv']
missing = [name for name in required if not (raw_dir/name).exists()]
if missing:
    raise FileNotFoundError(f'Missing {missing} in {raw_dir}. Set ENABLE_LIVE_COLLECTION=True with configured API keys and permissions, or provide your own genuine CSVs. No fake dataset is loaded.')
prices = pd.read_csv(raw_dir/'prices.csv')
social = pd.read_csv(raw_dir/'social.csv')
news = pd.read_csv(raw_dir/'news.csv')
for label, frame in [('prices',prices),('social',social),('news',news)]:
    print(label, len(frame), 'rows; columns:', list(frame.columns))
print('Social sources:', social.source.value_counts().to_dict() if not social.empty else {})
print('Earliest social publication:', social.published_at.min() if not social.empty else 'not available')
display(prices.tail(3))

In [ ]:
if 'sentiment' not in social:
    social = label_sentiment(social)
    write_frame(social, raw_dir/'social.csv')
if 'sentiment' not in news:
    news = label_sentiment(news)
    write_frame(news, raw_dir/'news.csv')
print('Sentiment range:', (social.sentiment.min(),social.sentiment.max()) if not social.empty else None)
print('Missing publication timestamps:', pd.to_datetime(social.published_at,utc=True,errors='coerce').isna().sum())

## 4 — Point-in-time features and ground truth

Prediction target: **next session return** `close[t+1] / close[t] - 1`. At the 16:00 HKT close of session `t` we may use only posts already published; anything after close is assigned to the next *observed* session. The code handles market weekends via observed price dates. **It does not yet handle HKEX half-day closes** — see limitations.

In [ ]:
frame = build_features(prices,social,news)
display(frame[['session','close','lihkg_mentions','futu_mentions','news_mentions','social_sentiment','source_count','target_close']].tail(12))
usable = frame.loc[(frame.source_count==2)&frame.target_return.notna()]
print(f'Two-source labeled sessions: {len(usable)} / {len(frame)}. Training gate needs >= 60 *fully observed* sessions.')

## 5 — Train and validate (only if sufficient genuine observations)

Two otherwise-identical HistGradientBoosting regressors are compared: **price-only** vs **price+sentiment**. The last 20% is reserved as a never-touched test, one supervised row is purged between train and test, and earlier data is used for time-series CV when possible. A **no-change** benchmark predicts `next_close = current_close`. Training stops rather than fabricating a result if fewer than 60 eligible sessions exist.

In [ ]:
try:
    evaluation, trained = train_evaluate(frame, min_rows=60)
except ValueError as exc:
    evaluation, trained = None, None
    print('TRAINING NOT RUN:', exc)
    print('Collect more *historical, timestamped, permitted* sentiment from BOTH sources. Ten recent days alone are insufficient.')
else:
    save_artifacts(evaluation, trained, out_dir)
    report_path = render_html(evaluation, SELECTED_TICKER, out_dir / 'report.html')
    metrics_table = pd.DataFrame([{'model': k, **v['test']} for k,v in evaluation['models'].items()])
    display(metrics_table)
    print('Saved held-out evaluation and chart:',report_path)

## 6 — Prediction for the latest observed market session

Only returns a **real model forecast** when a trained model exists and *both* social sources have posts assigned to the latest session. Otherwise it reports unavailable. Support and resistance are historical rolling high/low estimates, **not certainty or trade advice**.

In [ ]:
latest = frame.iloc[-1]
summary = summarize_ticker(asset,frame,trained['price_plus_sentiment'] if trained else None)
display(pd.DataFrame([summary]).T.rename(columns={0:'value'}))

## 7 — Per-exchange visual leaderboard

After gathering the remaining four assets (repeat `news-predictor collect/train` for each), generate the exchange report. Assets lacking eligible trained models display **unavailable** forecasts rather than invented values. The CLI command `news-predictor report` performs this across `config/assets.json`.

In [ ]:
# Current stock is shown here. For all five HK tickers use: news-predictor report
leaderboard_path=render_leaderboard([summary], ROOT/'outputs'/'current_ticker_report.html')
print('Interactive HTML + CSV:', leaderboard_path,leaderboard_path.with_suffix('.csv'))

## 8 — Checklist before sending to recruiter

- [ ] Two actual social data sources permitted for training; Futu source officially authorized (ideally both via Apify as requested).
- [ ] Five assets per required exchange; confirm whether Korea is mandatory.
- [ ] Ten days of raw social posts available; train/testing history independently long enough.
- [ ] Source UTC timestamps and available engagement/follower fields recorded, not fabricated.
- [ ] Exa news fetched with original publication timestamps.
- [ ] Trained model and correct chronological backtest on genuine observations.
- [ ] Sorted visual report, support/resistance, sentiment, direction and position per ticker.
- [ ] GitHub link and non-secret Actor IDs/run settings. **Never send real tokens in a public repository.**

Source/reference URLs and unresolved blockers: [`docs/SOURCES.md`](../docs/SOURCES.md).